# Hierarchical B-Splines (HB-splines)

This notebook gives a self-contained, step-by-step introduction to
**Hierarchical B-splines (HB-splines)** and their use in 1-D adaptive
finite element analysis.

HB-splines organise B-spline basis functions into **levels of resolution**.
When a region needs finer resolution, the coarse-level functions whose support
overlaps that region are *deactivated* and replaced by their **children** at
the next (dyadically refined) level.

Unlike their truncated counterpart (THB-splines), HB-splines are **not
truncated**, so they **do not form a partition of unity** after non-uniform
refinement.  This is the key limitation that THB-splines fix by applying
a truncation operator.

**Contents**

| Section | Topic |
|---------|-------|
| 1 | Univariate B-splines: `BsplineSpace` |
| 2 | Dyadic refinement and children |
| 3 | Hierarchical B-spline space |
| 4 | Partition of unity — what HB-splines lose |
| 5 | Stiffness matrix assembly |
| 6 | Solving a smooth 1-D diffusion problem |
| 7 | Adaptive FEM: boundary layer |
| 8 | Convergence: adaptive vs uniform |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse.linalg as spla
import sys, pathlib

_project_root = pathlib.Path('..').resolve()
sys.path.insert(0, str(_project_root))

from HBSplines.src.bspline_space  import BsplineSpace
from HBSplines.src.hb_space       import HBsplineSpace
from HBSplines.src.assembly       import hb_stiffness_matrix, hb_load_vector_full, hb_solve
from HBSplines.src.estimator      import local_residual, global_residual
from HBSplines.src.marking        import dorfler_mark, max_mark
from HBSplines                    import adaptive_solve, AdaptiveSolverSettings
from HBSplines.problems           import smooth_diffusion, boundary_layer_right, interior_layer

%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 170,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'figure.facecolor': 'white',
    'axes.facecolor': '#fbfbf8',
    'axes.edgecolor': '#262626',
    'axes.linewidth': 0.9,
    'axes.titlesize': 14,
    'axes.titleweight': 'semibold',
    'axes.labelsize': 11,
    'axes.grid': True,
    'grid.color': '#d9d4cb',
    'grid.linewidth': 0.7,
    'grid.alpha': 0.45,
    'grid.linestyle': '--',
    'font.size': 11,
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'stix',
    'legend.frameon': False,
    'legend.fontsize': 9,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})

LINE_COLORS  = ['#0f4c5c', '#e36414', '#6a994e', '#7b2cbf', '#c1121f', '#1d3557']
SURFACE_CMAP = 'viridis'
ERROR_CMAP   = 'magma'


def style_axes(ax, *, square=False, xlabel=None, ylabel=None, title=None):
    if xlabel is not None: ax.set_xlabel(xlabel)
    if ylabel is not None: ax.set_ylabel(ylabel)
    if title  is not None: ax.set_title(title, pad=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if square: ax.set_aspect('equal', adjustable='box')
    return ax


---
## 1. Univariate B-splines

A **B-spline** $B_{i,p}$ of degree $p$ is defined by a *local knot vector*
$\xi = [\xi_0, \xi_1, \ldots, \xi_{p+1}]$ of length $p+2$ via the
Cox–de Boor recursion:

$$
B_{\xi,0}(x) = \begin{cases}1 & \xi_0 \le x < \xi_{p+1} \\ 0 & \text{otherwise}\end{cases}
$$
$$
B_{\xi,p}(x) = \frac{x-\xi_0}{\xi_p-\xi_0}\,B_{\xi_{[:-1]},p-1}(x)
              + \frac{\xi_{p+1}-x}{\xi_{p+1}-\xi_1}\,B_{\xi_{[1:]},p-1}(x)
$$

`BsplineSpace` wraps a **full clamped knot vector** and exposes per-function
evaluation (`eval`), first and higher derivatives (`deriv(x, i, r=1)`), and
the **partition of unity** property $\sum_i B_i(x)=1$ on the parameter domain.

> The Cox–de Boor implementation is shared with the LR-spline package:
> `LRSplines.src.lr_basis._eval1d` and `_deriv1d`.


In [ ]:
# ── A complete quadratic B-spline basis on [0, 1] ────────────────────────────
knots_demo = [0, 0, 0, 0.25, 0.5, 0.75, 1, 1, 1]
sp_demo    = BsplineSpace(knots_demo, degree=2)

x = np.linspace(0, 1, 300)
B = sp_demo.eval_all(x)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.0), constrained_layout=True)

# — left: basis functions
ax = axes[0]
for i in range(sp_demo.dim):
    ax.plot(x, B[:, i], lw=1.8, color=LINE_COLORS[i % len(LINE_COLORS)],
            label=f'$B_{{{i},2}}$')
style_axes(ax, xlabel='$x$', ylabel='$B_{{i,2}}(x)$',
           title=f'Quadratic B-spline basis  (dim = {sp_demo.dim})')
ax.legend(ncol=3, loc='upper center', fontsize=9)
ax.set_ylim(-0.05, 1.05)

# — right: partition of unity
ax = axes[1]
pou = B.sum(axis=1)
ax.plot(x, pou, lw=2.2, color=LINE_COLORS[0])
ax.axhline(1.0, color='#aaa', lw=0.9, ls='--')
style_axes(ax, xlabel='$x$', ylabel=r'$\sum_i B_i(x)$',
           title='Partition of Unity  (single level)')
ax.set_ylim(0.95, 1.05)
print(f'Max deviation from 1: {np.abs(pou - 1).max():.2e}')


In [ ]:
# ── First and second derivatives ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.0), constrained_layout=True)

for i in range(sp_demo.dim):
    c = LINE_COLORS[i % len(LINE_COLORS)]
    axes[0].plot(x, sp_demo.deriv(x, i, r=1), lw=1.6, color=c, label=f"$B'_{{{i}}}$")
    axes[1].plot(x, sp_demo.deriv(x, i, r=2), lw=1.6, color=c, label=f"$B''_{{{i}}}$")

for ax, r, title in zip(axes, [1, 2],
                         ["First derivative $B'_{i,2}$", "Second derivative $B''_{i,2}$"]):
    style_axes(ax, xlabel='$x$', ylabel=title, title=title)
    ax.legend(ncol=3, loc='upper center', fontsize=9)
    ax.axhline(0, color='#aaa', lw=0.8, ls='--')


---
## 2. Dyadic refinement and children

**Dyadic refinement** inserts the midpoint of every knot interval, doubling
the number of unique breakpoints:

$$
\Xi_\ell \;\xrightarrow{\text{insert midpoints}}\; \Xi_{\ell+1}
$$

A basis function $B_{i,p}$ at level $\ell$ has **children** at level $\ell+1$:
all functions $B_{j,p}^{\ell+1}$ whose support is fully contained inside the
support of $B_{i,p}^\ell$.  For an interior function of degree $p$ there are
exactly $p+2$ children; boundary functions with repeated clamping knots may
have fewer.


In [ ]:
# ── Dyadic refinement: coarse → fine ─────────────────────────────────────────
sp_coarse = BsplineSpace([0, 0, 0, 0.5, 1, 1, 1], degree=2)
sp_fine   = sp_coarse.refine()

print('Coarse knots:', sp_coarse.knots)
print('Fine   knots:', sp_fine.knots)
print(f'Coarse dim: {sp_coarse.dim}   Fine dim: {sp_fine.dim}')
print()
for i in range(sp_coarse.dim):
    ch = sp_coarse.get_children(i)
    lk = sp_coarse._local_knots(i)
    print(f'  B_{i} support [{lk[0]:.3f}, {lk[-1]:.3f}]  →  children at fine level: {ch}')


In [ ]:
# ── Visualise parent B_2 and its children ────────────────────────────────────
parent_idx = 2
children   = sp_coarse.get_children(parent_idx)

x_c = np.linspace(0, 1, 400)
fig, axes = plt.subplots(1, 2, figsize=(12.8, 3.8), constrained_layout=True)

# left: coarse
ax = axes[0]
for i in range(sp_coarse.dim):
    lw = 2.4 if i == parent_idx else 1.0
    alpha = 1.0 if i == parent_idx else 0.35
    ax.plot(x_c, sp_coarse.eval(x_c, i), lw=lw, alpha=alpha,
            color=LINE_COLORS[i % len(LINE_COLORS)])
ax.fill_between(x_c, sp_coarse.eval(x_c, parent_idx), alpha=0.18,
                color=LINE_COLORS[parent_idx % len(LINE_COLORS)])
style_axes(ax, xlabel='$x$', ylabel='$B_i$',
           title=f'Coarse level — $B_{{{parent_idx}}}$ highlighted')

# right: fine level children
ax = axes[1]
for i in range(sp_fine.dim):
    if i in children:
        ax.plot(x_c, sp_fine.eval(x_c, i), lw=2.2,
                color=LINE_COLORS[list(children).index(i) % len(LINE_COLORS)],
                label=f'child $B^{{fine}}_{{{i}}}$')
    else:
        ax.plot(x_c, sp_fine.eval(x_c, i), lw=0.8, alpha=0.25, color='#999')
style_axes(ax, xlabel='$x$', ylabel='$B^{{fine}}_j$',
           title=f'Fine level — children of $B_{{{parent_idx}}}$')
ax.legend()


---
## 3. Hierarchical B-spline space

`HBsplineSpace` stores the hierarchical refinement state as:

- `sp_lev[l]` — the `BsplineSpace` at level $l$
- `A[l]` — boolean mask of **active** functions at level $l$
- `D[l]` — boolean mask of **deactivated** functions at level $l$

The total DOF count is $\sum_l |A_l|$.

**Refinement** (`hspace.refine(level, indices)`):
1. Deactivate the selected functions at `level`.
2. Activate their children at `level + 1` (creating the level if needed).

**Boundary DOFs**: functions with index $0$ (left) or $n_l - 1$ (right) at
any level touch the domain boundary and carry Dirichlet data.


In [ ]:
# ── Build a hierarchical space and refine twice ───────────────────────────────
hspace = HBsplineSpace([0, 0, 0, 0.5, 1, 1, 1], degree=2)
print('=== Before refinement ===')
print(hspace.summary())

# Refine function 2 (support [0.5, 1.0])
hspace.refine(level=0, indices=np.array([2]))
print('\n=== After first refinement (B_2) ===')
print(hspace.summary())
print('Global DOFs:', hspace.global_dofs())


In [ ]:
# ── Visualise active basis functions after refinement ────────────────────────
x = np.linspace(0, 1, 400)
dofs = hspace.global_dofs()

fig, ax = plt.subplots(figsize=(9.0, 4.0), constrained_layout=True)
for k, (lev, i) in enumerate(dofs):
    label = f'level {lev}, $B^{{{lev}}}_{{{i}}}$'
    ax.plot(x, hspace.eval_dof(x, lev, i), lw=2.0,
            color=LINE_COLORS[k % len(LINE_COLORS)], label=label)

style_axes(ax, xlabel='$x$', ylabel='$B_k(x)$',
           title=f'Active HB-spline basis  ({hspace.nfuncs} DOFs, {hspace.nlevels} levels)')
ax.legend(fontsize=9, ncol=2)

# Mark the domain split
ax.axvline(0.5, color='#888', lw=0.9, ls='--')
ax.set_xlim(0, 1)


---
## 4. Partition of unity — what HB-splines lose

A single-level B-spline space satisfies $\sum_i B_i(x) = 1$ everywhere
(partition of unity, PoU).

After **adaptive refinement**, HB-splines **no longer form a PoU**: the
deactivated coarse-level functions are simply removed, leaving a "gap" in
the sum near the refined region.

This is why **THB-splines** apply a *truncation* operator: each surviving
coarse-level function is truncated by subtracting the fine-level
representations, restoring PoU.

> The PoU defect in HB-splines means:
> - Constant functions are **not** exactly representable.
> - The FEM bilinear form is still well-defined, but conditioning can degrade.


In [ ]:
# ── Partition of unity: single level vs after refinement ─────────────────────
hs_uniform = HBsplineSpace([0, 0, 0, 0.5, 1, 1, 1], degree=2)

hs_refined = HBsplineSpace([0, 0, 0, 0.5, 1, 1, 1], degree=2)
hs_refined.refine(level=0, indices=np.array([1, 2]))   # refine interior

x = np.linspace(0, 1, 400)

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.0), constrained_layout=True)

for ax, hs, title_str in zip(
    axes,
    [hs_uniform, hs_refined],
    ['No refinement (single level) — PoU = 1', 'After refinement — PoU violated'],
):
    B = hs.eval_all(x)
    pou = B.sum(axis=1)
    for k, (lev, i) in enumerate(hs.global_dofs()):
        ax.plot(x, B[:, k], lw=1.4, alpha=0.6,
                color=LINE_COLORS[k % len(LINE_COLORS)])
    ax2 = ax.twinx()
    ax2.plot(x, pou, lw=2.2, color='#c1121f', ls='--', label=r'$\sum B_i$')
    ax2.axhline(1.0, color='#888', lw=0.9)
    ax2.set_ylim(0.6, 1.15)
    ax2.set_ylabel(r'$\sum_i B_i(x)$', color='#c1121f')
    ax2.legend(loc='lower right')
    style_axes(ax, xlabel='$x$', ylabel='$B_k(x)$', title=title_str)

print('Uniform  — max |PoU - 1|:', np.abs(hs_uniform.eval_all(x).sum(axis=1) - 1).max())
print('Refined  — max |PoU - 1|:', np.abs(hs_refined.eval_all(x).sum(axis=1) - 1).max())


---
## 5. Stiffness matrix assembly

For the advection-diffusion problem
$$-m\,u'' + b\,u' = f \quad \text{on } [0,1], \qquad u(0)=u_0,\; u(1)=u_L$$
the **stiffness matrix** entries are

$$A_{ij} = \int_0^1 \bigl[m\,B_i'\,B_j' + b\,B_j'\,B_i\bigr]\,\mathrm{d}x$$

where $B_i$ is the **test function** (row) and $B_j$ is the **trial function** (column).

**Key implementation detail**: integration is performed *span-by-span* within
the intersection of the two supports, because $B_i'$ and $B_j'$ are
piecewise polynomials — integrating over the full support in a single
Gauss rule would straddle interior breakpoints and give wrong results.


In [ ]:
# ── Assemble stiffness matrix on a uniform space ──────────────────────────────
hspace_asm = HBsplineSpace([0, 0, 0, 0.25, 0.5, 0.75, 1, 1, 1], degree=2)
A = hb_stiffness_matrix(hspace_asm, m=1.0, b_adv=0.0)

print(f'Space DOFs   : {hspace_asm.nfuncs}')
print(f'Matrix shape : {A.shape}')
print(f'Non-zeros    : {A.nnz}')
print(f'Symmetric?   : max|A - Aᵀ| = {abs(A - A.T).max():.2e}  (0 for pure diffusion)')

fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0), constrained_layout=True)

axes[0].spy(A, markersize=6, color=LINE_COLORS[0])
style_axes(axes[0], title='Stiffness matrix sparsity pattern')
axes[0].spines['top'].set_visible(True)
axes[0].spines['right'].set_visible(True)

import matplotlib.colors as mcolors
im = axes[1].matshow(A.toarray(), cmap='RdBu_r',
                      norm=mcolors.CenteredNorm())
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
style_axes(axes[1], title='Stiffness matrix values')
axes[1].spines['top'].set_visible(True)
axes[1].spines['right'].set_visible(True)


---
## 6. Solving a smooth 1-D diffusion problem

We solve
$$-u'' = \pi^2 \sin(\pi x) \quad \text{on } [0,1], \qquad u(0) = u(1) = 0$$
whose exact solution is $u(x) = \sin(\pi x)$.

Dirichlet boundary conditions are imposed by identifying the active
**boundary DOFs** (functions with index 0 or $n_{\ell}-1$ at their level)
and restricting the linear system to the interior DOFs.


In [ ]:
# ── Solve smooth diffusion problem ───────────────────────────────────────────
prob = smooth_diffusion()   # -u'' = π² sin(πx),  u(0)=u(1)=0

knots_fine = [0,0,0, 0.125,0.25,0.375,0.5,0.625,0.75,0.875, 1,1,1]
hspace_sol = HBsplineSpace(knots_fine, degree=2)

c_full, A_sol, F_sol = hb_solve(hspace_sol, prob.f, prob.u0, prob.uL, prob.m, prob.b)

print(f'DOFs       : {hspace_sol.nfuncs}')
print(f'Interior   : {len(hspace_sol.interior_dofs())}')

x_plot = np.linspace(0, 1, 300)
u_h    = hspace_sol.eval_solution(x_plot, c_full)
u_ex   = prob.u_ex(x_plot)
err    = np.abs(u_h - u_ex)

L2  = np.sqrt(np.trapezoid((u_h - u_ex)**2, x_plot))
Linf = err.max()
print(f'L² error   : {L2:.4e}')
print(f'L∞ error   : {Linf:.4e}')

fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.0), constrained_layout=True)

ax = axes[0]
ax.plot(x_plot, u_ex, lw=2.2, color=LINE_COLORS[0], label='Exact $u$')
ax.plot(x_plot, u_h,  lw=1.8, color=LINE_COLORS[1], ls='--', label='HB-FEM $u_h$')
style_axes(ax, xlabel='$x$', ylabel='$u(x)$', title='Solution comparison')
ax.legend()

ax = axes[1]
ax.semilogy(x_plot, err + 1e-16, lw=1.8, color=LINE_COLORS[2])
style_axes(ax, xlabel='$x$', ylabel='$|u_h - u|$', title='Pointwise error (log scale)')


---
## 7. Adaptive FEM: boundary layer

We now solve the **right boundary layer** problem:
$$-m\,u'' + b\,u' = 0 \quad \text{on }[0,1],\qquad u(0)=0,\;u(1)=1$$
with exact solution $u(x) = (e^{(b/m)x}-1)/(e^{b/m}-1)$.

For $b/m = 10$, the solution is nearly flat on $[0, 0.8]$ and rises steeply
near $x = 1$ over a layer of width $O(m/b) = 0.1$.  Uniform refinement
resolves this poorly; **adaptive refinement** automatically concentrates
DOFs near $x = 1$.

The AFEM loop is:

$$
\textbf{Solve} \;\to\; \textbf{Estimate} \;\to\; \textbf{Mark} \;\to\; \textbf{Refine}
$$

- **Estimate**: cell-wise residual $\eta_j = h_j \|f + m\,u_h'' - b\,u_h'\|_{L^2(\text{cell})}$
- **Mark**: Dörfler (bulk) criterion — mark the smallest set with $\sum_{\text{marked}} \eta_j^2 \ge \theta\,\eta^2$
- **Refine**: deactivate marked functions and activate their children


In [ ]:
# ── Adaptive solve: right boundary layer ─────────────────────────────────────
prob_bl = boundary_layer_right(m=1.0, b=10.0)

hspace_bl = HBsplineSpace([0, 0, 0, 0.5, 1, 1, 1], degree=2)
settings   = AdaptiveSolverSettings(max_dofs=200, tol=1e-5, theta=0.4,
                                     strategy='dorfler', verbose=True)
result = adaptive_solve(prob_bl, hspace_bl, settings)

print(f'\nFinal DOFs  : {result.hspace.nfuncs}')
print(f'Levels      : {result.hspace.nlevels}')
print(f'Converged   : {result.converged}')


In [ ]:
# ── Plot adaptive solution and error ─────────────────────────────────────────
x_plot = np.linspace(0, 1, 400)
hsp    = result.hspace
c      = result.c_full

u_h  = hsp.eval_solution(x_plot, c)
u_ex = prob_bl.u_ex(x_plot)
err  = np.abs(u_h - u_ex)

fig, axes = plt.subplots(1, 3, figsize=(14.4, 4.0), constrained_layout=True)

# — solution
ax = axes[0]
ax.plot(x_plot, u_ex, lw=2.0, color=LINE_COLORS[0], label='Exact')
ax.plot(x_plot, u_h,  lw=1.6, color=LINE_COLORS[1], ls='--', label='HB-FEM')
style_axes(ax, xlabel='$x$', ylabel='$u(x)$', title='Boundary Layer Solution')
ax.legend()

# — pointwise error
ax = axes[1]
ax.semilogy(x_plot, err + 1e-16, lw=1.8, color=LINE_COLORS[2])
style_axes(ax, xlabel='$x$', ylabel='$|u_h - u|$', title='Pointwise Error (log scale)')

# — convergence history
ax = axes[2]
ax.semilogy(result.history_dofs, result.history_eta, 'o-',
            lw=1.8, ms=5, color=LINE_COLORS[0], label=r'$\eta$ (global estimator)')
style_axes(ax, xlabel='DOFs', ylabel=r'$\eta$', title='Convergence History')
ax.legend()

L2  = np.sqrt(np.trapezoid((u_h - u_ex)**2, x_plot))
print(f'L² error : {L2:.4e}')


In [ ]:
# ── Active functions per level after adaptive refinement ─────────────────────
hsp = result.hspace
dofs = hsp.global_dofs()

fig, ax = plt.subplots(figsize=(10.0, 4.0), constrained_layout=True)

level_colors = LINE_COLORS
for k, (lev, i) in enumerate(dofs):
    B_k = hsp.eval_dof(x_plot, lev, i)
    ax.fill_between(x_plot, B_k, alpha=0.25, color=level_colors[lev % len(level_colors)])
    ax.plot(x_plot, B_k, lw=1.2, color=level_colors[lev % len(level_colors)])

# Legend for levels
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=level_colors[l], alpha=0.5, label=f'Level {l}')
                   for l in range(hsp.nlevels)]
ax.legend(handles=legend_elements, loc='upper left')
style_axes(ax, xlabel='$x$', ylabel='$B_k(x)$',
           title=f'Active HB-spline functions by level  ({hsp.nfuncs} DOFs)')
ax.axvline(0.8, color='#888', lw=0.9, ls=':', label='steep region starts')


---
## 8. Convergence: adaptive vs uniform

We compare the $L^2$ error as a function of DOFs for:

- **Uniform refinement**: repeated dyadic refinement of the full space
- **Adaptive refinement**: the AFEM loop with Dörfler marking

For the smooth problem both strategies achieve the optimal
$O(h^{p+1}) = O(h^3)$ rate.

For the **boundary layer** problem, uniform refinement wastes DOFs in the
smooth region, giving suboptimal rates.  Adaptive refinement recovers the
optimal rate by concentrating DOFs near $x = 1$.


In [ ]:
# ── Convergence study — smooth diffusion ─────────────────────────────────────
import warnings
prob_smooth = smooth_diffusion()
x_conv = np.linspace(0, 1, 500)
u_ex_conv = prob_smooth.u_ex(x_conv)

def uniform_solve(n_cells):
    """Solve on uniform mesh with n_cells interior intervals (n_cells+1 knot spans)."""
    h = 1.0 / (n_cells + 1)
    knots = [0., 0., 0.] + list(np.arange(h, 1.0, h)) + [1., 1., 1.]
    hs = HBsplineSpace(knots, degree=2)
    c, _, _ = hb_solve(hs, prob_smooth.f, prob_smooth.u0, prob_smooth.uL,
                       prob_smooth.m, prob_smooth.b)
    u_h = hs.eval_solution(x_conv, c)
    return hs.nfuncs, np.sqrt(np.trapezoid((u_h - u_ex_conv)**2, x_conv))

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    uniform_results = [uniform_solve(n) for n in [2, 4, 8, 16, 32]]

dofs_u, errs_u = zip(*uniform_results)

fig, ax = plt.subplots(figsize=(7.0, 5.0), constrained_layout=True)
ax.loglog(dofs_u, errs_u, 'o-', lw=1.8, ms=6, color=LINE_COLORS[0], label='Uniform')

# Reference slope O(h³) ~ O(N^{-3/2}) for degree-2 on 1D
ns = np.array(dofs_u, dtype=float)
ax.loglog(ns, 0.5 * errs_u[0] * (ns / ns[0])**(-1.5), '--',
          lw=1.2, color='#888', label='$O(N^{-3/2})$  reference')
style_axes(ax, xlabel='DOFs (N)', ylabel='$L^2$ error', title='Convergence — smooth diffusion')
ax.legend()
print('Uniform DOFs:', list(dofs_u))
print('Uniform L2 : ', [f'{e:.3e}' for e in errs_u])


In [ ]:
# ── Convergence study — boundary layer ───────────────────────────────────────
import warnings
prob_bl2 = boundary_layer_right(m=1.0, b=10.0)
x_conv = np.linspace(0, 1, 500)
u_ex_bl = prob_bl2.u_ex(x_conv)

def uniform_solve_bl(n_cells):
    h = 1.0 / (n_cells + 1)
    knots = [0., 0., 0.] + list(np.arange(h, 1.0, h)) + [1., 1., 1.]
    hs = HBsplineSpace(knots, degree=2)
    c, _, _ = hb_solve(hs, prob_bl2.f, prob_bl2.u0, prob_bl2.uL,
                       prob_bl2.m, prob_bl2.b)
    u_h = hs.eval_solution(x_conv, c)
    return hs.nfuncs, np.sqrt(np.trapezoid((u_h - u_ex_bl)**2, x_conv))

def adaptive_solve_bl(max_dofs):
    hs = HBsplineSpace([0, 0, 0, 0.5, 1, 1, 1], degree=2)
    s  = AdaptiveSolverSettings(max_dofs=max_dofs, tol=0.0, theta=0.4,
                                 strategy='dorfler', verbose=False)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        res = adaptive_solve(prob_bl2, hs, s)
    u_h = hs.eval_solution(x_conv, res.c_full)
    return hs.nfuncs, np.sqrt(np.trapezoid((u_h - u_ex_bl)**2, x_conv))

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    unif_bl = [uniform_solve_bl(n) for n in [3, 7, 15, 31, 63]]
    adap_bl = [adaptive_solve_bl(d) for d in [10, 20, 40, 80, 160]]

dofs_u2, errs_u2 = zip(*unif_bl)
dofs_a,  errs_a  = zip(*adap_bl)

fig, ax = plt.subplots(figsize=(7.0, 5.0), constrained_layout=True)
ax.loglog(dofs_u2, errs_u2, 's-', lw=1.8, ms=6, color=LINE_COLORS[0], label='Uniform')
ax.loglog(dofs_a,  errs_a,  'o-', lw=1.8, ms=6, color=LINE_COLORS[1], label='Adaptive (Dörfler)')
ns = np.array(dofs_u2, dtype=float)
ax.loglog(ns, 2.0 * errs_u2[0] * (ns / ns[0])**(-1.5), '--',
          lw=1.2, color='#888', label='$O(N^{-3/2})$  reference')
style_axes(ax, xlabel='DOFs (N)', ylabel='$L^2$ error',
           title='Convergence — right boundary layer (b/m = 10)')
ax.legend()


---
## Summary

| Section | Topic | Key class / function |
|---------|-------|---------------------|
| 1 | Univariate B-splines, PoU | `BsplineSpace`, `.eval_all()`, `.deriv()` |
| 2 | Dyadic refinement, children | `.refine()`, `.get_children()` |
| 3 | Hierarchical space | `HBsplineSpace`, `.refine()`, `.global_dofs()` |
| 4 | Partition of unity defect | `.check_partition_of_unity()` |
| 5 | Stiffness matrix | `hb_stiffness_matrix()` |
| 6 | Smooth diffusion solve | `hb_solve()` |
| 7 | Adaptive FEM — boundary layer | `adaptive_solve()`, `AdaptiveSolverSettings` |
| 8 | Convergence: adaptive vs uniform | `smooth_diffusion()`, `boundary_layer_right()` |

**Key takeaway — HB vs THB**

| Property | HB-splines | THB-splines |
|----------|-----------|-------------|
| Partition of unity | ✗ lost after refinement | ✓ guaranteed by truncation |
| Implementation complexity | simpler | slightly more complex |
| FEM conditioning | can degrade | stable |
| Refinement locality | same | same |
| Implemented here | 1D advection-diffusion | 2D Poisson |
